In [115]:
# Basic libraries 
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pykalman import KalmanFilter
from itertools import combinations
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.stattools import coint
import statsmodels.api as sm
import pickle
import os
import sys

PROJECT_ROOT = os.path.abspath("..")
sys.path.insert(0, PROJECT_ROOT)

In [116]:
# Scripts 
from src.local_config import *
from src.cointegration import *
from src.kalman import *
from src.pairs import *
from src.trading_signal import *
from src.backtest import *
from src.plots import *
from src.utils import *
from src. portfolio_con import *

In [117]:
# Load in data from previous notebook
df1_is = pd.read_csv(PROJECT_ROOT / "data/df1_is",
                     index_col=0, parse_dates=True)
df1_oos = pd.read_csv(PROJECT_ROOT / "data/df1_oos",
                      index_col=0, parse_dates=True)

df2_is = pd.read_csv(PROJECT_ROOT / "data/df2_is",
                     index_col=0, parse_dates=True)
df2_oos = pd.read_csv(PROJECT_ROOT / "data/df2_oos",
                      index_col=0, parse_dates=True)

# Portfolio Construction

- Test for correlation when combining our pairs for trading to ensure that the portfolio is `diversified'.
- Then give a 50/50 weighting to each sector to ensure fairness in the portfolio when trading

## Correlation between pairs

Only do diagnostics on in-sample and then use the same pairs for out-of-sample portfolio construction (to prevent look-ahead bias from occurring).

In [118]:
# Load in results from previous notebook
static_results_df_is = pd.read_csv(PROJECT_ROOT / "data/static_hedge_ratio_is")
dynamic_results_df_is = pd.read_csv(PROJECT_ROOT / "data/dynamic_hedge_ratio_is")

In [119]:
pairs_is = [
    {"y": "1347 HK Equity", "x": "268 HK Equity", "df": df1_is},
    {"y": "857 HK Equity", "x": "2386 HK Equity", "df": df2_is},
    {"y": "3993 HK Equity", "x": "2689 HK Equity", "df": df2_is},
    {"y": "1258 HK Equity", "x": "3899 HK Equity", "df": df2_is},
    {"y": "1258 HK Equity", "x": "189 HK Equity", "df": df2_is},
]


In [120]:
spread_returns_dict = {}

for _, row in static_results_df_is.iterrows():
    pair   = row["pair"]
    y_name, x_name = [s.strip() for s in pair.split(" vs ")]
    df     = df1_is if y_name in df1_is.columns else df2_is
    spread = df[y_name] - row["beta"] * df[x_name]
    spread_returns_dict[pair] = spread.diff().dropna()

results = portfolio_diagnostics(spread_returns_dict)

In [121]:
print(results["correlation"].round(3))
print(results["initial_vif"].round(2))
print(results["final_vif"].round(2))
print("Retained:     ", results["retained_pairs"])
print("Corr dropped: ", results["corr_dropped"])
print("VIF dropped:  ", results["vif_dropped"])

                                  1347 HK Equity vs 268 HK Equity  \
1347 HK Equity vs 268 HK Equity                             1.000   
857 HK Equity vs 2386 HK Equity                             0.065   
3993 HK Equity vs 2689 HK Equity                            0.024   
1258 HK Equity vs 3899 HK Equity                            0.062   

                                  857 HK Equity vs 2386 HK Equity  \
1347 HK Equity vs 268 HK Equity                             0.065   
857 HK Equity vs 2386 HK Equity                             1.000   
3993 HK Equity vs 2689 HK Equity                            0.019   
1258 HK Equity vs 3899 HK Equity                            0.019   

                                  3993 HK Equity vs 2689 HK Equity  \
1347 HK Equity vs 268 HK Equity                              0.024   
857 HK Equity vs 2386 HK Equity                              0.019   
3993 HK Equity vs 2689 HK Equity                             1.000   
1258 HK Equity vs 3899 HK Eq

In [122]:
portfolio_is = [
    {"y": "1347 HK Equity", "x": "268 HK Equity", "df": df1_is},
    {"y": "857 HK Equity", "x": "2386 HK Equity", "df": df2_is},
    {"y": "3993 HK Equity", "x": "2689 HK Equity", "df": df2_is},
    {"y": "1258 HK Equity", "x": "3899 HK Equity", "df": df2_is},
]

portfolio_oos = [
    {"y": "1347 HK Equity", "x": "268 HK Equity", "df": df1_oos},
    {"y": "857 HK Equity", "x": "2386 HK Equity", "df": df2_oos},
    {"y": "3993 HK Equity", "x": "2689 HK Equity", "df": df2_oos},
    {"y": "1258 HK Equity", "x": "3899 HK Equity", "df": df2_oos},
]

with open(PROJECT_ROOT / "data/portfolio_is.pkl", "wb") as f:
    pickle.dump(portfolio_is, f)

with open(PROJECT_ROOT / "data/portfolio_oos.pkl", "wb") as f:
    pickle.dump(portfolio_oos, f)


In [123]:
portfolio = pd.DataFrame([
    {"pair": "1347 HK Equity vs 268 HK Equity",  "sector": "tech",      "weight": 0.50},
    {"pair": "857 HK Equity vs 2386 HK Equity",  "sector": "commodity", "weight": 1/6},
    {"pair": "3993 HK Equity vs 2689 HK Equity", "sector": "commodity", "weight": 1/6},
    {"pair": "1258 HK Equity vs 3899 HK Equity", "sector": "commodity", "weight": 1/6},
])

portfolio = portfolio.merge(
    static_results_df_is[["pair", "beta"]],
    on="pair"
)

portfolio = portfolio.rename(columns={"beta": "hedge_ratio"})

portfolio.to_csv(PROJECT_ROOT / "data/portfolio", index=False)
portfolio

,pair,sector,weight,hedge_ratio
0,1347 HK Equity vs 268 HK Equity,tech,0.500000,0.739501
1,857 HK Equity vs 2386 HK Equity,commodity,0.166667,0.854406
2,3993 HK Equity vs 2689 HK Equity,commodity,0.166667,1.290032
3,1258 HK Equity vs 3899 HK Equity,commodity,0.166667,0.961593
